In [2]:
# Cell 1：加载模型，确认结构
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import sys

sys.path.insert(0, "../src")

# 加载权重，只看 hparams 和 buffer 名字
ckpt = torch.load("../data/two_tower_v2_best.pth", map_location="cpu")
hp   = ckpt["hparams"]
print("hparams:", hp)
print("buffers:", [k for k in ckpt["model_state"] if not any(x in k for x in ["weight", "bias"])])

hparams: {'emb_dim': 32, 'tower_dims': [64, 32], 'n_users': 6040, 'n_movies': 3706}
buffers: ['genre_matrix', 'age_min', 'age_range']


In [3]:
# Cell 2：重建模型 + 加载权重
class TwoTowerV2(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=32, tower_dims=[64,32],
                 genre_matrix=None, age_min=0.0, age_range=1.0):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users, emb_dim)
        self.occ_emb   = nn.Embedding(21, 16)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)

        n_genre  = genre_matrix.shape[1] if genre_matrix is not None else 18
        user_in  = emb_dim + 16 + 1 + 1   # 50
        item_in  = emb_dim + n_genre       # 50

        self.user_tower = nn.Sequential(
            nn.Linear(user_in, tower_dims[0]), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(tower_dims[0], tower_dims[1])
        )
        self.movie_tower = nn.Sequential(
            nn.Linear(item_in, tower_dims[0]), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(tower_dims[0], tower_dims[1])
        )

        if genre_matrix is not None:
            self.register_buffer("genre_matrix", genre_matrix)
        self.register_buffer("age_min",   torch.tensor(age_min,   dtype=torch.float32))
        self.register_buffer("age_range", torch.tensor(age_range, dtype=torch.float32))

    def encode_user(self, user_ids, genders, ages, occupations):
        age_norm = (ages.float() - self.age_min) / self.age_range
        x = torch.cat([
            self.user_emb(user_ids),
            self.occ_emb(occupations),
            genders.float().unsqueeze(-1),
            age_norm.unsqueeze(-1)
        ], dim=-1)
        return self.user_tower(x)

    def encode_item(self, movie_ids):
        genre = self.genre_matrix[movie_ids]
        x = torch.cat([self.movie_emb(movie_ids), genre], dim=-1)
        return self.movie_tower(x)

    def forward(self, user_ids, movie_ids, genders, ages, occupations):
        u = self.encode_user(user_ids, genders, ages, occupations)
        v = self.encode_item(movie_ids)
        return torch.sigmoid((u * v).sum(dim=-1))

# 加载
genre_matrix = ckpt["model_state"]["genre_matrix"]
model = TwoTowerV2(
    n_users=hp["n_users"], n_movies=hp["n_movies"],
    emb_dim=hp["emb_dim"], tower_dims=hp["tower_dims"],
    genre_matrix=genre_matrix
)
model.load_state_dict(ckpt["model_state"])
model.eval()

# 验证：随便跑一个 forward
with torch.no_grad():
    score = model(
        torch.tensor([0]),   # user_id
        torch.tensor([0]),   # movie_id
        torch.tensor([1]),   # gender
        torch.tensor([25]),  # age
        torch.tensor([4])    # occupation
    )
print("forward OK, score:", score.item())

forward OK, score: 0.82927405834198


In [4]:
# Cell 3：导出 user tower 为 ONNX
import torch.onnx
import os

# 单独把 user tower 包成一个 Module，方便导出
class UserTowerONNX(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.user_emb  = model.user_emb
        self.occ_emb   = model.occ_emb
        self.user_tower = model.user_tower
        self.register_buffer("age_min",   model.age_min)
        self.register_buffer("age_range", model.age_range)

    def forward(self, user_ids, genders, ages, occupations):
        age_norm = (ages.float() - self.age_min) / self.age_range
        x = torch.cat([
            self.user_emb(user_ids),
            self.occ_emb(occupations),
            genders.float().unsqueeze(-1),
            age_norm.unsqueeze(-1)
        ], dim=-1)
        return self.user_tower(x)

user_tower_onnx = UserTowerONNX(model).eval()

# 示例输入（batch_size=1）
dummy = (
    torch.tensor([0]),   # user_ids
    torch.tensor([1]),   # genders
    torch.tensor([25]),  # ages
    torch.tensor([4]),   # occupations
)

os.makedirs("../data", exist_ok=True)
out_path = "../data/user_tower.onnx"

torch.onnx.export(
    user_tower_onnx,
    dummy,
    out_path,
    input_names=["user_ids", "genders", "ages", "occupations"],
    output_names=["user_vec"],
    dynamic_axes={
        "user_ids":     {0: "batch_size"},
        "genders":      {0: "batch_size"},
        "ages":         {0: "batch_size"},
        "occupations":  {0: "batch_size"},
        "user_vec":     {0: "batch_size"},
    },
    opset_version=17,
)
print("导出成功：", out_path)
print("文件大小：", round(os.path.getsize(out_path) / 1024, 1), "KB")

/var/folders/lv/mnjxv0d53_xczbffr49m5z_40000gn/T/ipykernel_2162/3251281632.py:38: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0409 21:14:08.287000 2162 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `UserTowerONNX([...]` with `torch.export.export(..., strict=False)`...


/opt/anaconda3/envs/mlenv/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Obtain model graph for `UserTowerONNX([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
导出成功： ../data/user_tower.onnx
文件大小： 2.8 KB


/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


In [5]:
# Cell 4：验证 ONNX 输出和 PyTorch 一致
import onnxruntime as ort

# PyTorch 结果
with torch.no_grad():
    pt_vec = user_tower_onnx(
        torch.tensor([0]),
        torch.tensor([1]),
        torch.tensor([25]),
        torch.tensor([4])
    ).numpy()

# ONNX Runtime 结果
sess = ort.InferenceSession("../data/user_tower.onnx")
ort_vec = sess.run(["user_vec"], {
    "user_ids":    np.array([0], dtype=np.int64),
    "genders":     np.array([1], dtype=np.int64),
    "ages":        np.array([25], dtype=np.int64),
    "occupations": np.array([4], dtype=np.int64),
})[0]

print("PyTorch 输出:", pt_vec)
print("ONNX    输出:", ort_vec)
print("最大误差:", np.abs(pt_vec - ort_vec).max())

PyTorch 输出: [[-0.2651992   0.09196822 -0.3394887  -0.09659474  0.13560186  0.22505768
  -0.0482168  -0.19733593 -0.07233601 -0.01233684 -0.11296152  0.28664765
  -0.19998424  0.2873819  -0.13840556  0.16743939  0.24006881 -0.11261262
   0.231505    0.13767105 -0.00559632  0.00901826  0.2339255   0.2933532
  -0.09235067 -0.09291965 -0.07497321 -0.17918617 -0.03013018  0.10874328
  -0.29583764  0.23900588]]
ONNX    输出: [[-0.26519924  0.09196822 -0.33948866 -0.09659474  0.13560185  0.22505769
  -0.0482168  -0.19733594 -0.07233602 -0.01233683 -0.1129615   0.28664768
  -0.19998427  0.28738192 -0.13840556  0.16743939  0.24006882 -0.11261262
   0.23150502  0.13767105 -0.00559635  0.00901827  0.23392549  0.2933532
  -0.09235065 -0.09291967 -0.07497322 -0.17918618 -0.03013018  0.10874329
  -0.29583764  0.23900592]]
最大误差: 4.4703484e-08


In [6]:
# Cell 5：推理延迟 benchmark
import time

N = 2000  # 跑 2000 次取平均

# PyTorch
pt_times = []
with torch.no_grad():
    for _ in range(N):
        t0 = time.perf_counter()
        user_tower_onnx(
            torch.tensor([0]),
            torch.tensor([1]),
            torch.tensor([25]),
            torch.tensor([4])
        )
        pt_times.append(time.perf_counter() - t0)

# ONNX Runtime
ort_inputs = {
    "user_ids":    np.array([0],  dtype=np.int64),
    "genders":     np.array([1],  dtype=np.int64),
    "ages":        np.array([25], dtype=np.int64),
    "occupations": np.array([4],  dtype=np.int64),
}
ort_times = []
for _ in range(N):
    t0 = time.perf_counter()
    sess.run(["user_vec"], ort_inputs)
    ort_times.append(time.perf_counter() - t0)

pt_ms  = np.mean(pt_times)  * 1000
ort_ms = np.mean(ort_times) * 1000
print(f"PyTorch      均值: {pt_ms:.4f} ms")
print(f"ONNX Runtime 均值: {ort_ms:.4f} ms")
print(f"加速比: {pt_ms / ort_ms:.2f}x")

PyTorch      均值: 0.0542 ms
ONNX Runtime 均值: 0.0067 ms
加速比: 8.06x


In [7]:
# Cell 6：Netron 可视化
import netron
netron.start("../data/user_tower.onnx", browse=True)


Serving '../data/user_tower.onnx' at http://localhost:8080


('localhost', 8080)

In [8]:
# Cell 7：查看当前 recommender.py
with open("../src/recommender.py") as f:
    print(f.read())

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import faiss
from pathlib import Path

# ── 模型定义 ──────────────────────────────────────────────────
class TwoTowerV2(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=32, tower_dims=[64, 32], n_occ=21):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users, emb_dim)
        self.occ_emb   = nn.Embedding(n_occ, 16)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        self.user_tower = nn.Sequential(
            nn.Linear(emb_dim + 16 + 1 + 1, tower_dims[0]), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(tower_dims[0], tower_dims[1])
        )
        self.movie_tower = nn.Sequential(
            nn.Linear(emb_dim + 18, tower_dims[0]), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(tower_dims[0], tower_dims[1])
        )
        self.register_buffer("genre_matrix", torch.zeros(n_movies, 18))
        self.register_buffer("age_min",   torch.tensor(0.0))
        self.r

In [9]:
# Cell 8：写入新的 recommender.py
new_code = '''import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import faiss
import os
from pathlib import Path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ── 模型定义 ──────────────────────────────────────────────────
class TwoTowerV2(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=32, tower_dims=[64, 32], n_occ=21):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users, emb_dim)
        self.occ_emb   = nn.Embedding(n_occ, 16)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        self.user_tower = nn.Sequential(
            nn.Linear(emb_dim + 16 + 1 + 1, tower_dims[0]), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(tower_dims[0], tower_dims[1])
        )
        self.movie_tower = nn.Sequential(
            nn.Linear(emb_dim + 18, tower_dims[0]), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(tower_dims[0], tower_dims[1])
        )
        self.register_buffer("genre_matrix", torch.zeros(n_movies, 18))
        self.register_buffer("age_min",   torch.tensor(0.0))
        self.register_buffer("age_range", torch.tensor(1.0))

    def get_item_vec(self, movie_idx):
        emb   = self.movie_emb(movie_idx)
        genre = self.genre_matrix[movie_idx]
        return self.movie_tower(torch.cat([emb, genre], dim=-1))

    def get_user_vec(self, user_idx, gender, age, occ):
        age_norm = (age - self.age_min) / self.age_range
        emb      = self.user_emb(user_idx)
        o_emb    = self.occ_emb(occ)
        return self.user_tower(torch.cat([emb, o_emb, gender, age_norm], dim=-1))


# ── 核心推理类 ────────────────────────────────────────────────
class Recommender:
    def __init__(self, data_dir: str, model_path: str, onnx_path: str = None):
        data_dir   = Path(data_dir)
        model_path = Path(model_path)

        # 加载数据
        ratings = pd.read_csv(data_dir / "ml-1m/ratings.dat", sep="::",
            names=["user_id","movie_id","rating","timestamp"], engine="python")
        self.movies = pd.read_csv(data_dir / "ml-1m/movies.dat", sep="::",
            names=["movie_id","title","genres"], engine="python", encoding="latin-1")
        self.users  = pd.read_csv(data_dir / "ml-1m/users.dat", sep="::",
            names=["user_id","gender","age","occupation","zip"], engine="python")

        # index 体系
        all_users  = sorted(ratings["user_id"].unique())
        all_movies = sorted(ratings["movie_id"].unique())
        self.user2idx  = {u: i for i, u in enumerate(all_users)}
        self.movie2idx = {m: i for i, m in enumerate(all_movies)}
        self.idx2movie = {i: m for m, i in self.movie2idx.items()}

        # 每个用户看过的电影
        self.watched = ratings.groupby("user_id")["movie_id"].apply(set).to_dict()

        # 加载 PyTorch 模型（item tower 始终用 PyTorch，只跑一次）
        ckpt = torch.load(model_path, map_location="cpu")
        hp   = ckpt["hparams"]
        self.model = TwoTowerV2(n_users=hp["n_users"], n_movies=hp["n_movies"],
                                emb_dim=hp["emb_dim"], tower_dims=hp["tower_dims"])
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()

        # ONNX Runtime（user tower 热路径）
        self.ort_sess = None
        onnx_path = onnx_path or str(data_dir / "user_tower.onnx")
        if Path(onnx_path).exists():
            import onnxruntime as ort
            self.ort_sess = ort.InferenceSession(onnx_path)
            print(f"user tower: ONNX Runtime ({onnx_path})")
        else:
            print("user tower: PyTorch (ONNX not found, fallback)")

        # 建 Faiss 索引（item tower，只跑一次）
        all_idx = torch.arange(len(self.movie2idx))
        with torch.no_grad():
            item_vecs = self.model.get_item_vec(all_idx).numpy().astype("float32")
        faiss.normalize_L2(item_vecs)
        self.index = faiss.IndexFlatIP(item_vecs.shape[1])
        self.index.add(item_vecs)

        print(f"Recommender ready — {len(self.user2idx)} users, "
              f"{self.index.ntotal} items in Faiss index")

    def _get_user_vec(self, user_idx: int, gender: float, age: float, occ: int) -> np.ndarray:
        """返回 shape (1, 32) float32 numpy 数组"""
        if self.ort_sess is not None:
            return self.ort_sess.run(["user_vec"], {
                "user_ids":    np.array([user_idx], dtype=np.int64),
                "genders":     np.array([int(gender)], dtype=np.int64),
                "ages":        np.array([int(age)],    dtype=np.int64),
                "occupations": np.array([occ],         dtype=np.int64),
            })[0].astype("float32")
        else:
            with torch.no_grad():
                return self.model.get_user_vec(
                    user_idx = torch.tensor([user_idx]),
                    gender   = torch.tensor([[gender]]),
                    age      = torch.tensor([[age]]),
                    occ      = torch.tensor([occ]),
                ).numpy().astype("float32")

    def recommend(self, user_id: int, top_k: int = 10, recall_k: int = 50):
        if user_id not in self.user2idx:
            return {"error": f"user_id {user_id} not found"}

        u = self.users[self.users["user_id"] == user_id].iloc[0]
        gender_val = 0.0 if u["gender"] == "F" else 1.0

        u_vec = self._get_user_vec(
            user_idx = self.user2idx[user_id],
            gender   = gender_val,
            age      = float(u["age"]),
            occ      = int(u["occupation"]),
        )
        faiss.normalize_L2(u_vec)

        scores, indices = self.index.search(u_vec, recall_k)
        watched = self.watched.get(user_id, set())

        results = []
        for idx_, score in zip(indices[0], scores[0]):
            movie_id = self.idx2movie[idx_]
            if movie_id in watched:
                continue
            title = self.movies[self.movies["movie_id"] == movie_id]["title"].values[0]
            results.append({"movie_id": int(movie_id), "title": title, "score": round(float(score), 4)})
            if len(results) >= top_k:
                break

        return {"user_id": user_id, "recommendations": results}
'''

with open("../src/recommender.py", "w") as f:
    f.write(new_code)
print("写入完成")

写入完成


In [10]:
# Cell 9：验证新 recommender
import importlib, sys
for mod in list(sys.modules.keys()):
    if "recommender" in mod:
        del sys.modules[mod]

from recommender import Recommender
rec = Recommender(data_dir="../data", model_path="../data/two_tower_v2_best.pth")
print(rec.recommend(1))

Loading faiss.
Successfully loaded faiss.
Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes.


user tower: ONNX Runtime (../data/user_tower.onnx)
Recommender ready — 6040 users, 3706 items in Faiss index
{'user_id': 1, 'recommendations': [{'movie_id': 2396, 'title': 'Shakespeare in Love (1998)', 'score': 0.4878}, {'movie_id': 3175, 'title': 'Galaxy Quest (1999)', 'score': 0.4319}, {'movie_id': 2858, 'title': 'American Beauty (1999)', 'score': 0.4247}, {'movie_id': 34, 'title': 'Babe (1995)', 'score': 0.4077}, {'movie_id': 2683, 'title': 'Austin Powers: The Spy Who Shagged Me (1999)', 'score': 0.3954}, {'movie_id': 2291, 'title': 'Edward Scissorhands (1990)', 'score': 0.3869}, {'movie_id': 356, 'title': 'Forrest Gump (1994)', 'score': 0.3839}, {'movie_id': 1265, 'title': 'Groundhog Day (1993)', 'score': 0.3781}, {'movie_id': 1580, 'title': 'Men in Black (1997)', 'score': 0.3604}, {'movie_id': 39, 'title': 'Clueless (1995)', 'score': 0.3604}]}
